# Ungraded Lab: Capstone Project Lab

## Task 1: Exploratory Data Analysis and Feature Engineering
Begin by understanding your data and creating meaningful features that capture energy consumption patterns.

<b>Step 1:</b>  Initial Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load and examine the data
df = pd.read_csv('smart_energy_data_v2.csv')


# Perform comprehensive EDA including:
# - Data quality assessment
# - Distribution analysis
# - Correlation studies


def explore_energy_data(df):
    """
    Conduct thorough exploratory analysis
    Returns: Dictionary of analysis results
    """
    # Your EDA code here
    pass

<b>Step 2:</b> Feature Engineering

In [ ]:
# Create advanced features capturing:
# - Usage patterns
# - Environmental impacts
# - Household characteristics
# Consider both supervised and unsupervised learning needs

def engineer_features(df):
    """
    Create advanced features for energy analysis
    Returns: DataFrame with new features
    """
    # Your feature engineering code here
    pass

## Task 2: Model Development
Implement both supervised and unsupervised learning approaches to understand energy consumption patterns.

<b>Step 1:</b> Regression Model

In [ ]:
# Develop a model to predict daily energy consumption
# Include:
# - Train test splitting
# - Feature selection
# - Model training
# - Performance evaluation
# - Cross-validation
# Note: Remember to assess whether the selected model requires feature scaling.

def build_consumption_model(df, target_col='energy_consumption', test_size=0.2, random_state=42):
    """
    Create and train energy consumption prediction model
    Returns: Trained model, features and performance metrics
    """
    # Your regression model code here
    pass

<b>Step 2:</b> Clustering Analysis

In [ ]:
# Implement clustering to identify usage patterns
# Consider:
# - Feature scaling
# - Optimal cluster selection
# - Pattern interpretation

def analyze_usage_patterns(X):
    """
    Perform clustering analysis on energy usage data
    Returns: Cluster assignments, model and features
    """
    # Your clustering code here
    pass

## Task 3: Model Deployment
Deploy your models using AWS Services, ensuring efficient processing and monitoring.

In [ ]:
# Set up batch processing deployment
# Include:
# Error handling
# Model saving into S3
# Model loading

def deploy_energy_models(models, config):
    """
    Deploy models for production use by saving them into S3 and loading it for inference
    Returns: Loaded model ready for prediction    """
    # Your deployment code here
    pass

## Task 4: Results Analysis and Recommendations
Transform your technical results into actionable business insights.

In [ ]:
# Create comprehensive analysis including:
# - Performance metrics
# - Usage insights
# - Optimization recommendations
# - Visualization of key findings

def generate_insights(metrics, final_df, cluster_summary):
    """
    Create business insights from model results
    Returns: Dictionary of findings and recommendations
    """
    # Your analysis code here
    pass

## Solution Code
Need a hand or are curious to compare your approach? Below is a complete solution you can use as a reference. This is just one of many valid ways to solve the problem. Make sure to give it a try on your own first. Use this implementation to troubleshoot, learn new techniques, or confirm your logic. Keep experimenting and enjoy the process!

## Task 1: Exploratory Data Analysis and Feature Engineering Solution Code

<b>Step 1:</b>  Initial Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load and examine the data
df = pd.read_csv('smart_energy_data_v2.csv')


# Perform comprehensive EDA including:
# - Data quality assessment
# - Distribution analysis
# - Correlation studies

def explore_energy_data(df):
    """
    Conduct thorough exploratory analysis
    Returns: Dictionary of analysis results
    """
    analysis_results = {}

    # Basic data overview
    print("Basic Info:")
    print(df.info())
    print("\nMissing Values:")
    print(df.isnull().sum())
    print("\nSummary Statistics:")
    print(df.describe())

    # Data quality: missing values and duplicates
    analysis_results['missing_values'] = df.isnull().sum()
    analysis_results['duplicates'] = df.duplicated().sum()
    print(f"\nNumber of duplicated rows: {analysis_results['duplicates']}")

    # Distribution of numerical features
    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[col], kde=True, bins=30)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    # Correlation analysis
    plt.figure(figsize=(12, 8))
    corr = df[numeric_cols].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", square=True)
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.show()
    analysis_results['correlation_matrix'] = corr

    return analysis_results

results = explore_energy_data(df)

print(results['missing_values'])           # To check missing values per column
print(results['correlation_matrix'])       # To inspect the correlation matrix

<b>Step 2:</b> Feature Engineering

In [ ]:
# Create advanced features capturing:
# - Usage patterns
# - Environmental impacts
# - Household characteristics
# Consider both supervised and unsupervised learning needs

def engineer_features(df):
    """
    Create advanced features for energy analysis
    Returns: DataFrame with new features
    """
    df = df.copy()

    # Map month names to numbers and seasons
    month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
                 'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
    season_map = {
        'winter': ['Dec', 'Jan', 'Feb'],
        'spring': ['Mar', 'Apr', 'May'],
        'summer': ['Jun', 'Jul', 'Aug'],
        'fall': ['Sep', 'Oct', 'Nov']
    }

    # Convert month to numeric
    df['month_num'] = df['month'].map(month_map)

    # Create season feature
    def get_season(month):
        for season, months in season_map.items():
            if month in months:
                return season
        return np.nan

    df['season'] = df['month'].apply(get_season)

    # Usage patterns
    df['is_weekend'] = df['day_of_week'].isin(['Sat', 'Sun']).astype(int)
    df['work_from_home_ratio'] = df['work_from_home_days'] / 7

    # Environmental interactions
    df['discomfort_index'] = 0.5 * (df['temperature_out'] + 61.0 +
                                    ((df['temperature_out'] - 68.0) * 1.2) +
                                    (df['humidity_out'] * 0.094))

    df['solar_adjusted_usage'] = df['energy_consumption'] / (df['has_solar'] + 1)

    # Household characteristics
    df['usage_per_m2'] = df['energy_consumption'] / df['home_size']
    df['usage_per_device'] = df['energy_consumption'] / (df['device_count'] + 1)
    df['usage_per_child'] = df['energy_consumption'] / (df['num_children'] + 1)

    # Normalize usage by average daily usage
    df['relative_usage'] = df['energy_consumption'] / (df['avg_daily_usage'] + 1e-3)

    # Encode categorical variables for supervised models
    df = pd.get_dummies(df, columns=['region', 'energy_contract', 'season'], drop_first=True)

    return df

df_featurized = engineer_features(df)
df_featurized.head()

## Task 2: Model Development Solution Code

<b>Step 1:</b> Regression Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold
import numpy as np

# Develop a model to predict daily energy consumption
# Include:
# - Train test splitting
# - Feature selection
# - Model training
# - Performance evaluation
# - Cross-validation
# Note: Remember to assess whether the selected model requires feature scaling.
def build_consumption_model(df, target_col='energy_consumption', test_size=0.2, random_state=42):
    """
    Create and train energy consumption prediction model
    Returns: Trained model, features and performance metrics
    """
    # 1. Prepare target and features
    y = df[target_col]
    X = df.drop(columns=[target_col, 'user_id'], errors='ignore')
    X = pd.get_dummies(X, drop_first=True)

    # 2. Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # 3. Feature selection with RandomForest
    selector = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=random_state))
    selector.fit(X_train, y_train)
    selected_features = X_train.columns[selector.get_support()]

    X_train_sel = selector.transform(X_train)
    X_test_sel = selector.transform(X_test)

    # 4. Train model
    model = RandomForestRegressor(n_estimators=100, random_state=random_state)
    model.fit(X_train_sel, y_train)

    # 5. Simple 3-fold cross-validation R2 (just one metric, minimal)
    cv_r2 = cross_val_score(model, X_train_sel, y_train, cv=3, scoring='r2').mean()

    # 6. Test set evaluation
    y_pred = model.predict(X_test_sel)
    test_r2 = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # 7. Print metrics and selected features
    print(f"Selected features: {list(selected_features)}")
    print(f"CV R² (3-fold): {cv_r2:.3f}")
    print(f"Test R²: {test_r2:.3f}, Test RMSE: {test_rmse:.3f}")

    metrics = {
        'cv_r2': cv_r2,
        'test_r2': test_r2,
        'test_rmse': test_rmse,
        'selected_features': list(selected_features)
    }

    return model, metrics, selected_features

# Example call
consumption_model, metrics, selected_features_consumption_model = build_consumption_model(df_featurized)
print(metrics)

<b>Step 2:</b> Clustering Analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Implement clustering to identify usage patterns
# Consider:
# - Feature scaling
# - Optimal cluster selection
# - Pattern interpretation

def analyze_usage_patterns(X):
    """
    Perform clustering analysis on energy usage data
    Returns: Cluster assignments, model and features
    """
    # Select only numerical features
    X_num = X.select_dtypes(include=[np.number])
    clustering_features = X_num.columns

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_num)

    # Elbow method: inertia vs number of clusters
    inertias = []
    max_clusters = 10
    cluster_range = range(1, max_clusters + 1)
    for k in cluster_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X_scaled)
        inertias.append(kmeans.inertia_)

    # Plot the elbow curve
    plt.figure(figsize=(8, 4))
    plt.plot(cluster_range, inertias, marker='o')
    plt.title('Elbow Method: Inertia vs Number of Clusters')
    plt.xlabel('Number of Clusters')
    plt.ylabel('Inertia')
    plt.grid(True)
    plt.show()

    # Selecting the number of clusters
    optimal_k = 4

    # Train final model
    cluster_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    cluster_labels = cluster_model.fit_predict(X_scaled)

    # Cluster summary
    df_scaled = pd.DataFrame(X_scaled, columns=X_num.columns)
    df_scaled['cluster'] = cluster_labels
    cluster_summary = df_scaled.groupby('cluster').mean()

    print("\nCluster summary (mean scaled feature values):")
    print(cluster_summary)

    return cluster_labels, cluster_summary, cluster_model, clustering_features


cluster_labels, summary, cluster_model, clustering_features = analyze_usage_patterns(df_featurized)
df_featurized['cluster'] = cluster_labels

## Task 3: Model Deployment Solution Code
Deploy your models using AWS Services, ensuring efficient processing and monitoring.

In [ ]:
!pip install boto
import boto3
import pickle
import io
from botocore.exceptions import BotoCoreError, ClientError

# Set up batch processing deployment
# Include:
# Error handling
# Model saving into S3
# Model loading

def deploy_energy_models(models, config):
    """
    Deploy models for production use by saving them into S3 and loading it
    for inference
    Returns: Loaded model ready for prediction
    """
    s3 = boto3.client('s3')
    loaded_models = {}

    for name, model in models.items():
        try:
            # Save model to a bytes buffer
            buffer = io.BytesIO()
            pickle.dump(model, buffer)
            buffer.seek(0)

            s3_key = f"{config['model_prefix']}/{name}.pkl"
            s3.upload_fileobj(buffer, config['bucket'], s3_key)
            print(f"[INFO] Model '{name}' saved to s3://{config['bucket']}/{s3_key}")

        except (BotoCoreError, ClientError, pickle.PickleError) as e:
            print(f"[ERROR] Failed to save model '{name}': {e}")
            continue

        try:
            load_buffer = io.BytesIO()
            s3.download_fileobj(config['bucket'], s3_key, load_buffer)
            load_buffer.seek(0)
            loaded_model = pickle.load(load_buffer)

            loaded_models[name] = loaded_model
            print(f"[INFO] Model '{name}' loaded from S3")

        except (BotoCoreError, ClientError, pickle.PickleError) as e:
            print(f"[ERROR] Failed to load model '{name}': {e}")

    return loaded_models


# === Prepare Test Data ===

# Features used for consumption model and clustering
# (Assuming you already got these from previous function outputs)
X = df_featurized[selected_features_consumption_model]
y = df_featurized['energy_consumption']

train_idx, test_idx = train_test_split(df_featurized.index, test_size=0.2, random_state=42)
X_test = X.loc[test_idx]

X_cluster = df_featurized.loc[test_idx, clustering_features]
scaler = StandardScaler()
X_scaled_test = scaler.fit_transform(X_cluster)

# === Trained Models ===
models = {
    'consumption_model': consumption_model,
    'cluster_model': cluster_model
}

# === Config ===
config = {
    'bucket': 'power-nova-bucket',
    'model_prefix': 'energy_models'
}

# === Save & Load Models ===
deployed_models = deploy_energy_models(models, config)

# === Predict ===
consumption_preds = deployed_models['consumption_model'].predict(X_test)
cluster_labels = deployed_models['cluster_model'].predict(X_scaled_test)

# === Final Predictions DataFrame ===
final_df = df_featurized.loc[test_idx].copy()
final_df['predicted_energy_consumption'] = consumption_preds
final_df['cluster_label'] = cluster_labels

print(final_df.head())

## Task 4: Results Analysis and Recommendations Solution Code
Transform your technical results into actionable business insights.

In [ ]:
# Create comprehensive analysis including:
# - Performance metrics
# - Usage insights
# - Optimization recommendations
# - Visualization of key findings

def generate_insights(metrics, final_df, cluster_summary):
    """
    Create business insights from model results
    Returns: Dictionary of findings and recommendations
    """

    insights = {}

    # 1. Performance summary
    insights['performance_summary'] = (
        f"Model CV R²: {metrics.get('cv_r2', 0):.3f}, "
        f"Test R²: {metrics.get('test_r2', 0):.3f}, "
        f"Test RMSE: {metrics.get('test_rmse', 0):.3f}"
    )

    # 2. Cluster sizes and mean predicted consumption
    cluster_sizes = final_df['cluster_label'].value_counts().sort_index()
    cluster_mean_pred = final_df.groupby('cluster_label')['predicted_energy_consumption'].mean()

    insights['cluster_sizes'] = cluster_sizes.to_dict()
    insights['cluster_mean_predicted_consumption'] = cluster_mean_pred.to_dict()

    print("\nCluster Sizes:")
    print(cluster_sizes)

    print("\nAverage Predicted Energy Consumption per Cluster:")
    print(cluster_mean_pred)

    print("\nCluster Feature Summary (scaled means):")
    print(cluster_summary)

    # 3. Recommendations based on cluster consumption
    recommendations = []
    overall_mean = final_df['predicted_energy_consumption'].mean()
    for cluster, mean_cons in cluster_mean_pred.items():
        if mean_cons > overall_mean:
            recommendations.append(
                f"Cluster {cluster} shows higher than average predicted consumption — "
                "target energy-saving interventions here."
            )
        else:
            recommendations.append(
                f"Cluster {cluster} has lower predicted consumption — "
                "maintain current programs and monitor."
            )
    insights['recommendations'] = recommendations

    print("\nRecommendations:")
    for r in recommendations:
        print("- " + r)

    # 4. Visualizations
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    cluster_sizes.plot(kind='bar', color='skyblue')
    plt.title('Cluster Sizes')
    plt.xlabel('Cluster')
    plt.ylabel('Number of Samples')

    plt.subplot(1, 2, 2)
    cluster_mean_pred.plot(kind='bar', color='salmon')
    plt.title('Avg Predicted Energy Consumption per Cluster')
    plt.xlabel('Cluster')
    plt.ylabel('Energy Consumption')

    plt.tight_layout()
    plt.show()

    return insights

insights = generate_insights(metrics, final_df, summary)